# 02 — Feature Engineering

**Job:** turn labeled variants into the model-ready feature matrix, and
explore the *features themselves*.

**Contract:** reads `data/interim/labeled_snvs.parquet` → writes
`data/processed/feature_matrix.parquet`.

No re-parsing of the 3M-row raw file here — we start from notebook 01's
checkpoint. The "doing" logic lives in `clinvar_parse.py` and `features.py`;
this notebook orchestrates them and then does real EDA on the result.

## 01 — Imports and paths

In [ ]:
"""02_feature_engineering.ipynb — Feature Engineering for ClinVar missense variants."""
%load_ext autoreload
%autoreload 2

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from clinvar_parse import add_protein_change_columns
from features import extract_features

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = Path.cwd().parent / "data"
INTERIM_IN = DATA_DIR / "interim" / "labeled_snvs.parquet"
PROCESSED_OUT = DATA_DIR / "processed" / "feature_matrix.parquet"

RANDOM_STATE = 42
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

## 02 — Load the checkpoint

Dtypes come back exactly as written; no `low_memory` / `dtype=str` dance.

In [ ]:
df_labeled = pd.read_parquet(INTERIM_IN)
print(f"Loaded checkpoint: {len(df_labeled):,} labeled SNVs")

## 03 — Classify protein consequences, isolate missense

`add_protein_change_columns` adds `p_change` / `consequence` / `p_hgvs` and
prints the consequence funnel (synonymous / nonsense / frameshift / unknown
vs. the clean missense rows we keep).

In [ ]:
df_pc = add_protein_change_columns(df_labeled, name_col="Name")
df_missense = df_pc[df_pc["consequence"] == "missense"].copy()
print(f"\nMissense rows: {len(df_missense):,}")
print(df_missense["label"].value_counts().rename({0: "benign", 1: "pathogenic"}))

## 04 — Build the feature matrix (skip-and-count)

We extract the 8 **sequence-free** features. The two positional features
(`rel_position`, `seq_length`) are deliberately deferred — adding them would
require fetching UniProt sequences, introducing isoform-mismatch data loss and
a network dependency, for two features of uncertain value. We never silently
drop a row: parse failures are tallied.

In [ ]:
rows = []
skipped = {"parse_error": 0}
for _, r in df_missense.iterrows():
    try:
        feats = extract_features(r["p_hgvs"])   # no sequence -> 8 features
        feats["label"] = int(r["label"])
        feats["gene"] = r["GeneSymbol"]
        rows.append(feats)
    except Exception:
        skipped["parse_error"] += 1

In [ ]:
X_df = pd.DataFrame(rows)
feature_cols = [c for c in X_df.columns if c not in ("label", "gene")]
print(f"Built matrix: {X_df.shape[0]:,} variants × {len(feature_cols)} features "
      f"(skipped {skipped['parse_error']:,})")
print("Features:", feature_cols)
X_df.head()

## 05 — Class balance of the modeling set

Missense isolation can shift the balance vs. the full labeled set, so we
re-check it here on the actual matrix the models will see.

In [ ]:
print(X_df["label"].value_counts(normalize=True)
      .rename({0: "benign", 1: "pathogenic"}))

## 06 — Do the features carry signal? (group-by-class means)

Before any model: a fast descriptive check that the features differ by class.
We expect pathogenic variants to have **more negative** `blosum62` and
**larger** absolute chemical deltas. If they don't, the problem is upstream —
better to learn that here than after training.

In [ ]:
X_df.groupby("label")[feature_cols].mean().T

## 07 — Distribution of the headline feature by class

BLOSUM62 is expected to be the single most informative feature. Overlaid
histograms show *how* separable the classes are on it alone.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
for lbl, name, color in [(0, "benign", "#4C72B0"), (1, "pathogenic", "#C44E52")]:
    ax.hist(X_df.loc[X_df["label"] == lbl, "blosum62"],
            bins=range(-5, 5), alpha=0.6, label=name, color=color, density=True)
ax.set(xlabel="BLOSUM62 score", ylabel="density",
       title="BLOSUM62 by class (more negative = more disruptive)")
ax.legend()
plt.tight_layout()
plt.show()

## 08 — Feature correlations

Redundant features (e.g. signed vs. absolute deltas) will show up here. Useful
context for interpreting the linear model's coefficients in notebook 03.

In [ ]:
# %%
corr = X_df[feature_cols].corr()
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr, vmin=-1, vmax=1, cmap="RdBu_r")
ax.set_xticks(range(len(feature_cols)))
ax.set_xticklabels(feature_cols, rotation=90, fontsize=8)
ax.set_yticks(range(len(feature_cols)))
ax.set_yticklabels(feature_cols, fontsize=8)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_title("Feature correlation")
plt.tight_layout()
plt.show()

## 09 — Write the processed checkpoint

The model-ready matrix (features + `label` + `gene`) goes to `processed/`.
`gene` travels with it because notebook 03 needs it for the gene-held-out
leakage check.

In [ ]:
PROCESSED_OUT.parent.mkdir(parents=True, exist_ok=True)
X_df.to_parquet(PROCESSED_OUT, index=False)
print(f"Wrote {len(X_df):,} × {X_df.shape[1]} → "
      f"{PROCESSED_OUT.relative_to(PROJECT_ROOT)}")